In [ ]:
tabla='qtsop10'
col_tabla = "solopefec"
dat= "dat_ceqx001_essi"
col_essi='fec_sol'
essi='essi_dat_cqx001_etl'

In [ ]:
from decouple import config
from sqlalchemy import create_engine,text
import pandas as pd
from datetime import datetime, timedelta
import time 
import oracledb
import sys
import psycopg2
import numpy as np

In [ ]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [ ]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [5]:
fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

c1= text(query)
connection2.execute(c1)

#INICIO DEL ESSI

In [7]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

base1 = pd.read_sql_query(f"SELECT * FROM {tabla} WHERE {col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection0)



connection0.close()

In [8]:
print(base1.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68485 entries, 0 to 68484
Data columns (total 63 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   solopeoricenasicod          68485 non-null  object        
 1   solopecenasicod             68485 non-null  object        
 2   solopenum                   68485 non-null  int64         
 3   solopefec                   68485 non-null  datetime64[ns]
 4   solopeactmednum             68485 non-null  int64         
 5   solopemedtipdocidenpercod   68485 non-null  object        
 6   solopemedperasisdocidennum  68485 non-null  object        
 7   solopeinfmed                68485 non-null  object        
 8   solopesolfec                68485 non-null  object        
 9   solopeprofec                42189 non-null  object        
 10  solopeprohor                37610 non-null  object        
 11  estfiscod                   45872 non-null  object    

In [9]:
base1.columns

Index(['solopeoricenasicod', 'solopecenasicod', 'solopenum', 'solopefec',
       'solopeactmednum', 'solopemedtipdocidenpercod',
       'solopemedperasisdocidennum', 'solopeinfmed', 'solopesolfec',
       'solopeprofec', 'solopeprohor', 'estfiscod', 'estsopcod',
       'solopeestregcod', 'solopeusucrecod', 'solopecrefec', 'solopeusumodcod',
       'solopemodfec', 'solopecenquicod', 'solopesalopecod', 'solopeesttpo',
       'solopearehoscod', 'solopeservhoscod', 'solopeordnum', 'solopeemecod',
       'solopesolarehoscod', 'solopesolservhoscod', 'conopecod',
       'solopeactmedopenum', 'priopecod', 'riequicod', 'solopediashosprecan',
       'solopediashosposcan', 'solopepredetdes', 'solopereqadides',
       'motsopcod', 'solopetipanecod', 'soloperesexalabflg', 'soloperiequiflg',
       'soloperieneuflg', 'solopeconinfflg', 'solopeordtraflg',
       'solopeevalpqxinf', 'solopeevalpqxflg', 'solopeevalpqxfec',
       'solopeevalpqxoricenasicod', 'solopeevalpqxcenasicod',
       'solopeeval

In [10]:
base1 = base1.rename(columns={
    'solopeoricenasicod': 'ori_cas',
    'solopecenasicod': 'cod_cas',
    'solopenum': 'num_sol',
    'solopefec': 'fec_sol',
    'solopeactmednum': 'act_med',
    'solopemedtipdocidenpercod': 'cod_tdi',
    'solopemedperasisdocidennum': 'num_doc',
    'solopesolfec': 'fec_sop',
    'solopeprofec': 'fec_pro',
    'solopeprohor': 'hor_pro',
    'estfiscod': 'est_fis',
    'estsopcod': 'est_sop',
    'solopeestregcod': 'est_reg',
    'solopeusucrecod': 'usu_cre',
    'solopecrefec': 'fec_cre',
    'solopeusumodcod': 'usu_mod',
    'solopemodfec': 'fec_mod',
    'solopecenquicod': 'cod_cqx',
    'solopesalopecod': 'cod_sal',
    'solopeesttpo': 'est_ocu',
    'solopearehoscod': 'cod_are',
    'solopeservhoscod': 'cod_ser',
    'solopeordnum': 'ord_ope',
    'solopeemecod': 'eme_sol',
    'solopesolarehoscod': 'are_sol',
    'solopesolservhoscod': 'ser_sol',
    'conopecod': 'tip_ope',
    'solopeactmedopenum': 'act_med_ope',
    'priopecod': 'pri_ope',
    'riequicod': 'rie_qx',
    'solopediashosprecan': 'dia_hos_pre',
    'solopediashosposcan': 'dia_hos_pos',
    'motsopcod': 'mot_sus',
    'solopetipanecod': 'tip_ane',
    'soloperesexalabflg': 'res_lab',
    'soloperiequiflg': 'res_rie_qx',
    'soloperieneuflg': 'res_rie_neu',
    'solopeconinfflg': 'con_inf',
    'solopeordtraflg': 'ord_tra',
    'solopeevalpqxflg': 'res_eva',
    'solopeevalpqxfec': 'fec_eva',
    'solopeevalpqxoricenasicod': 'ori_cas_eva',
    'solopeevalpqxcenasicod': 'cod_cas_eva',
    'solopeevalpqxactmednum': 'act_med_eva',
    'solopeatesecnum': 'num_sec',
    'solopebuspacsecnum': 'pac_sec',
    'solopesolexafec': 'fec_sol_exa',
    'soloperesexalabfec': 'fec_res_exa',
    'soloperiequifec': 'fec_rie_car',
    'soloperieneufec': 'fec_rie_neu',
    'solopeevalpqxmedtipdoc': 'eva_tip_doc',
    'solopeevalpqxmeddocnum': 'eva_num_doc',
    'solopediashospreflg': 'sdp_hos_pre',
    'solopediashosposflg': 'sdp_hos_pos',
    'solopereqprotflg': 'req_pro_flg',
    'solopetopemecod': 'ser_eme_sol'
})

In [11]:
base1.shape

(68485, 63)

In [12]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [13]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 79 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   ori_cas         10 non-null     object             
 1   cod_cas         10 non-null     object             
 2   des_cas         10 non-null     object             
 3   cod_red         10 non-null     object             
 4   des_red         10 non-null     object             
 5   num_sol         10 non-null     float64            
 6   fec_sol         10 non-null     object             
 7   act_med         10 non-null     float64            
 8   cod_tdi         10 non-null     object             
 9   num_doc         10 non-null     object             
 10  fec_sop         10 non-null     object             
 11  fec_pro         0 non-null      object             
 12  hor_pro         0 non-null      object             
 13  est_fis         0 non-null      object

#TRAEMOS TODOS LOS MAESTROS

In [14]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
#id_red,cod_cas,des_cas,cod_red,des_red

cqxestfis= pd.read_sql_query(f"SELECT cod_efi,des_efi FROM dim_cqxestfis", con=connection2)
cqxestfis['cod_efi']=cqxestfis['cod_efi'].str.strip()

cqxestsolope= pd.read_sql_query(f"SELECT cod_eso,des_eso FROM dim_cqxestsolope", con=connection2)
cqxestsolope['cod_eso']=cqxestsolope['cod_eso'].str.strip()

cqxestreg= pd.read_sql_query(f"SELECT cod_est,des_est FROM dim_cqxestreg", con=connection2)
cqxestreg['cod_est']=cqxestreg['cod_est'].str.strip()

cqxcenqx= pd.read_sql_query(f"SELECT ori_cas,cod_cas,cod_cqx,des_cqx FROM dim_cqxcenqx", con=connection2)
cqxcenqx['cod_cqx']=cqxcenqx['cod_cqx'].str.strip()
cqxcenqx['cod_cas']=cqxcenqx['cod_cas'].str.strip()

cqxsalas= pd.read_sql_query(f"SELECT cod_ori,cod_cas,cod_cqx,cod_sal,des_sal FROM dim_cqxsalas", con=connection2)
cqxsalas['cod_sal']=cqxsalas['cod_sal'].str.strip()

areas = pd.read_sql_query(f"SELECT cod_are,des_are FROM dim_areas", con=connection2)
areas['cod_are']=areas['cod_are'].str.strip()

servicios = pd.read_sql_query(f"SELECT cod_ser,des_ser FROM dim_servicios", con=connection2)
servicios['cod_ser']=servicios['cod_ser'].str.strip()

emecod= pd.read_sql_query(f"SELECT cod_eme,des_eme FROM dim_emecod", con=connection2)
emecod['cod_eme']=emecod['cod_eme'].str.strip()

cqxtipope= pd.read_sql_query(f"SELECT cod_ope,des_ope FROM dim_cqxtipope", con=connection2)
cqxtipope['cod_ope']=cqxtipope['cod_ope'].str.strip()

cqxrieqx= pd.read_sql_query(f"SELECT cod_rqx,des_rqx FROM dim_cqxrieqx", con=connection2)
cqxrieqx['cod_rqx']=cqxrieqx['cod_rqx'].str.strip()

cqxmotsus= pd.read_sql_query(f"SELECT cod_msu,des_msu FROM dim_cqxmotsus", con=connection2)
cqxmotsus['cod_msu']=cqxmotsus['cod_msu'].str.strip()

cqxaneste= pd.read_sql_query(f"SELECT cod_ane,des_ane FROM dim_cqxaneste", con=connection2)
cqxaneste['cod_ane']=cqxaneste['cod_ane'].str.strip()

cqxmopreslab= pd.read_sql_query(f"SELECT cod_res,des_lab FROM dim_cqxmopreslab", con=connection2)
cqxmopreslab['cod_res']=cqxmopreslab['cod_res'].str.strip()

cqxmoprieqx= pd.read_sql_query(f"SELECT cod_rie,des_rie FROM dim_cqxmoprieqx", con=connection2)
cqxmoprieqx['cod_rie']=cqxmoprieqx['cod_rie'].str.strip()

cqxmoprieneu= pd.read_sql_query(f"SELECT cod_rieneu,des_rieneu FROM dim_cqxmoprieneu", con=connection2)
cqxmoprieneu['cod_rieneu']=cqxmoprieneu['cod_rieneu'].str.strip()

cqxmopconinf= pd.read_sql_query(f"SELECT cod_coninf,des_coninf FROM dim_cqxmopconinf", con=connection2)
cqxmopconinf['cod_coninf']=cqxmopconinf['cod_coninf'].str.strip()

cqxmopordtra= pd.read_sql_query(f"SELECT cod_ordtra,des_ordtra FROM dim_cqxmopordtra", con=connection2)
cqxmopordtra['cod_ordtra']=cqxmopordtra['cod_ordtra'].str.strip()

cqxresevapreqx= pd.read_sql_query(f"SELECT cod_reseva,des_reseva FROM dim_cqxresevapreqx", con=connection2)
cqxresevapreqx['cod_reseva']=cqxresevapreqx['cod_reseva'].str.strip()

In [15]:
a=base1.copy()

In [16]:
# base1=a

In [17]:
base1=pd.merge(base1,cas_red,how='left',on='cod_cas')
base1=base1.drop("id_red", axis=1)
base1.shape

(68485, 66)

In [18]:
base1['est_fis']=base1['est_fis'].str.strip()
base1=pd.merge(base1,cqxestfis,how='left',left_on='est_fis',right_on='cod_efi')
base1=base1.rename(columns={"des_efi":"des_fis"})
base1 = base1.drop("cod_efi", axis=1)
base1.shape

(68485, 67)

In [19]:
base1['est_sop']=base1['est_sop'].str.strip()
base1=pd.merge(base1,cqxestsolope,how='left',left_on='est_sop',right_on='cod_eso')
base1=base1.rename(columns={"des_eso":"des_sop"})
base1 = base1.drop("cod_eso", axis=1)
base1.shape

(68485, 68)

In [20]:
base1['est_reg']=base1['est_reg'].str.strip()
base1=pd.merge(base1,cqxestreg,how='left',left_on='est_reg',right_on='cod_est')
base1=base1.rename(columns={"des_est":"des_reg"})
base1 = base1.drop("cod_est", axis=1)
base1.shape

(68485, 69)

In [21]:
cqxcenqx["KEY"]=cqxcenqx["ori_cas"].str.strip()+cqxcenqx["cod_cas"].str.strip()+cqxcenqx["cod_cqx"].str.strip()
cqxcenqx=cqxcenqx.drop(["ori_cas",'cod_cas','cod_cqx'], axis=1)
base1["KEY"]=base1["ori_cas"].str.strip()+base1["cod_cas"].astype(str).str.strip()+base1['cod_cqx'].astype(str).str.strip()
base1['cod_sal']=base1['cod_sal'].str.strip()
base1=pd.merge(base1,cqxcenqx,how='left',on='KEY')
base1 = base1.drop("KEY", axis=1)
cqxcenqx = cqxcenqx.drop("KEY", axis=1)
base1.shape

(68485, 70)

In [22]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 70 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [23]:
cqxsalas["KEY"]=cqxsalas["cod_ori"].str.strip()+cqxsalas["cod_cas"].str.strip()+cqxsalas["cod_cqx"].str.strip()+cqxsalas["cod_sal"].str.strip()
cqxsalas=cqxsalas.drop(["cod_ori",'cod_cas','cod_cqx','cod_sal'], axis=1)
base1["KEY"]=base1["ori_cas"].str.strip()+base1["cod_cas"].astype(str).str.strip()+base1['cod_cqx'].astype(str).str.strip()+base1["cod_sal"].str.strip()
base1=pd.merge(base1,cqxsalas,how='left',on='KEY')
base1 = base1.drop("KEY", axis=1)
cqxsalas = cqxsalas.drop("KEY", axis=1)
base1.shape

(68485, 71)

In [24]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 71 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [25]:
base1=pd.merge(base1,areas,how='left',on='cod_are')
base1.shape

(68485, 72)

In [26]:
base1=pd.merge(base1,servicios,how='left',on='cod_ser')
base1.shape

(68485, 73)

In [27]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 73 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [28]:
base1['eme_sol']=base1['eme_sol'].str.strip()
base1=pd.merge(base1,emecod,how='left',left_on='eme_sol',right_on='cod_eme')
base1=base1.rename(columns={"des_eme":"des_eme_sol"})
base1 = base1.drop("cod_eme", axis=1)
base1.shape

(68485, 74)

In [29]:
base1['are_sol']=base1['are_sol'].str.strip()
areas=areas.rename(columns={"des_are":"des_are_sol"})
areas=areas.rename(columns={"cod_are":"are_sol"})
base1=pd.merge(base1,areas,how='left',on='are_sol')
base1.shape

(68485, 75)

In [30]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 75 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [31]:
base1['ser_sol']=base1['ser_sol'].str.strip()
servicios=servicios.rename(columns={"des_ser":"des_ser_sol"})
servicios=servicios.rename(columns={"cod_ser":"ser_sol"})

base1=pd.merge(base1,servicios,how='left',on='ser_sol')
base1.shape

(68485, 76)

In [32]:
base1.shape

(68485, 76)

In [33]:
base1=pd.merge(base1,cqxtipope,how='left',left_on='tip_ope',right_on='cod_ope')
base1=base1.rename(columns={"des_ope":"des_tip_ope"})
base1 = base1.drop( "cod_ope", axis=1)
base1.shape

(68485, 77)

In [34]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 77 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [35]:
base1=pd.merge(base1,cqxrieqx,how='left',left_on='rie_qx',right_on='cod_rqx')
base1 = base1.drop("cod_rqx", axis=1)
base1.shape

(68485, 78)

In [36]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 78 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [37]:
base1=pd.merge(base1,cqxmotsus,how='left',left_on='mot_sus',right_on='cod_msu')
base1=base1.rename(columns={"des_msu":"des_mot_sus"})
base1 = base1.drop("cod_msu", axis=1)
base1.shape

(68485, 79)

In [38]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 79 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [39]:
base1=pd.merge(base1,cqxaneste,how='left',left_on='tip_ane',right_on='cod_ane')
base1=base1.rename(columns={"des_ane":"des_tip_ane"})
base1 = base1.drop( "cod_ane", axis=1)
base1.shape

(68485, 80)

In [40]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 80 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [41]:
base1=pd.merge(base1,cqxmopreslab,how='left',left_on='res_lab',right_on='cod_res')
base1=base1.rename(columns={"des_lab":"des_res_lab"})
base1 = base1.drop( "cod_res", axis=1)
base1.shape

(68485, 81)

In [42]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 81 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [43]:
base1=pd.merge(base1,cqxmoprieqx,how='left',left_on='res_rie_qx',right_on='cod_rie')
base1=base1.rename(columns={"des_rie":"des_res_rieqx"})
base1 = base1.drop( "cod_rie", axis=1)
base1.shape

(68485, 82)

In [44]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 82 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [45]:
base1=pd.merge(base1,cqxmoprieneu,how='left',left_on='res_rie_neu',right_on='cod_rieneu')
base1=base1.rename(columns={"des_rieneu":"des_res_rieneu"})
base1 = base1.drop("cod_rieneu", axis=1)
base1.shape

(68485, 83)

In [46]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 83 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [47]:
base1=pd.merge(base1,cqxmopconinf,how='left',left_on='con_inf',right_on='cod_coninf')
base1=base1.rename(columns={"des_coninf":"des_con_inf"})
base1 = base1.drop("cod_coninf", axis=1)
base1.shape

(68485, 84)

In [48]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 84 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [49]:
base1=pd.merge(base1,cqxmopordtra,how='left',left_on='ord_tra',right_on='cod_ordtra')
base1=base1.rename(columns={"des_ordtra":"des_ord_tra"})
base1 = base1.drop("cod_ordtra", axis=1)
base1.shape

(68485, 85)

In [50]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 85 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [51]:
base1=pd.merge(base1,cqxresevapreqx,how='left',left_on='res_eva',right_on='cod_reseva')
base1=base1.rename(columns={"des_reseva":"des_res_eva"})
base1 = base1.drop("cod_reseva", axis=1)
base1.shape

(68485, 86)

In [52]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68485 entries, 0 to 68484
Data columns (total 86 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ori_cas           68485 non-null  object        
 1   cod_cas           68485 non-null  object        
 2   num_sol           68485 non-null  int64         
 3   fec_sol           68485 non-null  datetime64[ns]
 4   act_med           68485 non-null  int64         
 5   cod_tdi           68485 non-null  object        
 6   num_doc           68485 non-null  object        
 7   solopeinfmed      68485 non-null  object        
 8   fec_sop           68485 non-null  object        
 9   fec_pro           42189 non-null  object        
 10  hor_pro           37610 non-null  object        
 11  est_fis           45872 non-null  object        
 12  est_sop           68485 non-null  object        
 13  est_reg           68485 non-null  object        
 14  usu_cre           6848

In [59]:
base2.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'num_sol',
       'fec_sol', 'act_med', 'cod_tdi', 'num_doc', 'fec_sop', 'fec_pro',
       'hor_pro', 'est_fis', 'des_fis', 'est_sop', 'des_sop', 'est_reg',
       'des_reg', 'usu_cre', 'fec_cre', 'usu_mod', 'fec_mod', 'cod_cqx',
       'des_cqx', 'cod_sal', 'des_sal', 'est_ocu', 'cod_are', 'des_are',
       'cod_ser', 'des_ser', 'ord_ope', 'eme_sol', 'des_eme_sol', 'are_sol',
       'des_are_sol', 'ser_sol', 'des_ser_sol', 'tip_ope', 'des_tip_ope',
       'act_med_ope', 'pri_ope', 'rie_qx', 'des_rqx', 'dia_hos_pre',
       'dia_hos_pos', 'mot_sus', 'des_mot_sus', 'tip_ane', 'des_tip_ane',
       'res_lab', 'des_res_lab', 'res_rie_qx', 'des_res_rieqx', 'res_rie_neu',
       'des_res_rieneu', 'con_inf', 'des_con_inf', 'ord_tra', 'des_ord_tra',
       'res_eva', 'des_res_eva', 'fec_eva', 'ori_cas_eva', 'cod_cas_eva',
       'act_med_eva', 'num_sec', 'pac_sec', 'fec_sol_exa', 'fec_res_exa',
       'fec_rie_car', 'fec_rie_neu', 

In [60]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [61]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

{'solopeevalpqxinf',
 'solopeinfmed',
 'solopepredetdes',
 'solopereqadides',
 'solopereqprotdes',
 'solopetieprotflg',
 'solopetlffamnum'}

In [62]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [64]:
base.head()

,req_pro_flg,act_med,des_eme_sol,cod_red,est_reg,fec_res_exa,dia_hos_pre,mot_sus,rie_qx,eva_num_doc,...,act_med_ope,are_sol,num_sol,res_lab,ord_tra,est_sop,fec_sol,fec_mod,des_ser,des_fis
0,None,21708,NaN,17,1,None,NaN,None,1,None,...,1417545.0,05,520,None,None,2,2023-06-15,2023-06-16 10:10:50,CIRUGIA GENERAL ...,(I) PACIENTE SANO ...
1,None,6140286,NaN,10,1,None,NaN,None,1,None,...,NaN,05,3800,None,None,0,2025-10-21,2021-10-21 16:48:32,HEMATOLOGIA CLINICA ...,(III) PACIENTE CON ENFERMEDAD SISTEMICA SEVER...
2,None,6380753,NaN,10,1,None,NaN,None,1,None,...,NaN,05,5325,None,None,0,2023-06-08,2023-06-08 12:46:46,ORTOPEDIA Y TRAUMATOLOGIA ...,(I) PACIENTE SANO ...
3,None,6708687,NaN,05,1,None,NaN,None,1,None,...,6799114.0,05,3828,None,None,2,2023-05-09,2023-05-09 10:29:46,OTORRINOLARINGOLOGIA ...,(I) PACIENTE SANO ...
4,None,6799154,NaN,07,1,None,NaN,None,1,None,...,6836335.0,05,5182,None,None,2,2029-07-10,2019-12-11 13:01:49,GINECOLOGIA Y OBSTETRICIA ...,(I) PACIENTE SANO ...


In [55]:
base.columns

Index(['req_pro_flg', 'act_med', 'des_eme_sol', 'cod_red', 'est_reg',
       'fec_res_exa', 'dia_hos_pre', 'mot_sus', 'rie_qx', 'eva_num_doc',
       'ord_ope', 'pri_ope', 'act_med_eva', 'est_ocu', 'ori_cas', 'usu_cre',
       'fec_eva', 'ser_sol', 'ori_cas_eva', 'des_reg', 'des_cqx',
       'fec_rie_car', 'des_cas', 'des_sal', 'des_res_rieqx', 'fec_cre',
       'fec_sol_exa', 'sdp_hos_pos', 'res_rie_neu', 'hor_pro', 'pac_sec',
       'des_are', 'des_ser_sol', 'tip_ane', 'tip_ope', 'eva_tip_doc',
       'cod_cas_eva', 'num_sec', 'cod_ser', 'usu_mod', 'eme_sol', 'con_inf',
       'des_mot_sus', 'des_res_lab', 'res_eva', 'fec_sop', 'fec_rie_neu',
       'dia_hos_pos', 'des_red', 'des_are_sol', 'des_tip_ane', 'des_con_inf',
       'cod_cqx', 'des_res_eva', 'sdp_hos_pre', 'cod_tdi', 'est_fis',
       'des_ord_tra', 'des_sop', 'ser_eme_sol', 'fec_pro', 'cod_sal',
       'des_rqx', 'num_doc', 'des_res_rieneu', 'res_rie_qx', 'des_tip_ope',
       'cod_are', 'cod_cas', 'act_med_ope', 'are_sol'

In [57]:
basex=base1.copy()

In [65]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [66]:
base.to_sql(name=f'{essi}', con=connection1, if_exists='append', index=False,chunksize=10000)

6485

In [67]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3)
print("Proceso 1.2 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.2 completado satisfactoriamente en 386.545 segundos


In [ ]:
connection1.close()
connection2.close()
connection3.close()